# Assignment 3 – RAG Knowledge Assistant (Solution)
Input document: **troubleshooting_runbook.pdf** (Contoso Platform Troubleshooting Runbook).
Style follows `RAG_1.ipynb`: LangChain + Gemini + FAISS, answers returned as **1. Answer / 2. Supporting Evidence**.

Pipeline: PDF → text → chunks → embeddings → FAISS → top-k retrieval → grounded Gemini answer → Q&A app.

## Setup

In [ ]:
!pip install langchain langchain-google-genai langchain-community langchain-core langchain-text-splitters pypdf faiss-cpu

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
import os
import getpass

In [ ]:
os.environ["GEMINI_API_KEY"]=getpass.getpass("Enter your API Key")

In [ ]:
llm=ChatGoogleGenerativeAI(
    model='gemini-3.6-flash',
    temperature=0 #deterministic answers -> less hallucination
)

## Part 1 — Data understanding & preprocessing

In [ ]:
from langchain_community.document_loaders import PyPDFLoader #helps in reading the source data which is a pdf file

# Upload troubleshooting_runbook.pdf to /content in Colab (or keep it next to the notebook locally)
PDF_PATH="/content/troubleshooting_runbook.pdf" if os.path.exists("/content/troubleshooting_runbook.pdf") else "troubleshooting_runbook.pdf"
loader=PyPDFLoader(PDF_PATH)

In [ ]:
documents=loader.load()

report=""
for page in documents:
  report += page.page_content + '\n'

print("Total number of pages",len(documents))
print("Total characters",len(report))

In [ ]:
print(report[:1000]) #first 1000 characters from the document

### Text-quality checks
Page footers (`Contoso Platform Engineering | ... | Page N`) repeat on every page and would pollute chunks, so we strip them.

In [ ]:
import re

for d in documents:
  txt=d.page_content
  print(f"page {d.metadata['page']+1}: {len(txt):5d} chars | {len(txt.split()):4d} words | footer present: {'Synthetic demo data' in txt}")

FOOTER=re.compile(r"Contoso Platform Engineering\s*\|[\s\S]*?Synthetic demo data\s*(Page\s*\d+)?")
for d in documents:
  d.page_content=FOOTER.sub(" ",d.page_content).strip()

print("\nFooter left after cleaning:",any('Synthetic demo data' in d.page_content for d in documents))

### Chunking
The runbook is short (~2.4k words), so we use `chunk_size=1000` characters with `chunk_overlap=200` (same as `RAG_1.ipynb`) – this keeps one error-code section (symptom/causes/diagnostics/fix) mostly inside one chunk. Splitting prefers paragraph breaks, so section headings stay with their text.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter #helps in chunking

text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000, #max characters in each chunk
    chunk_overlap=200 #characters overlapping between consecutive chunks
)

chunks=text_splitter.split_documents(documents)
print("Number of chunks:",len(chunks))
print("Avg chunk length:",sum(len(c.page_content) for c in chunks)//len(chunks))
print("\nSample chunk (page",chunks[10].metadata['page']+1,"):\n",chunks[10].page_content)

## Part 2 — Embeddings & vector database
- **Embedding model:** `gemini-embedding-2` (Google Gemini) – converts every chunk into a dense vector so semantically similar text is close in vector space.
- **Vector store:** FAISS (in-memory, exact nearest-neighbour search) – ideal for a small corpus like this; each vector is stored with its chunk text and page metadata.

In [ ]:
embeddings=GoogleGenerativeAIEmbeddings(
    model='gemini-embedding-2'
)

In [ ]:
from langchain_community.vectorstores import FAISS
vector_store=FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)
print("Vectors stored:",vector_store.index.ntotal)

In [ ]:
query="What causes ERR-401 and how do I fix it?"

results=vector_store.similarity_search_with_score(query=query,k=3)

for i, (doc,score) in enumerate(results):
  print(f"RESULT{i}")
  print(f"Score (L2 distance, lower = closer):{score}")
  print(f"Page:{doc.metadata['page']+1}")
  print(f"Document:{doc.page_content[:400]}\n")

## Part 3 — Retrieval pipeline
Query → embedding → top-k chunks. We experiment with k = 3, 5, 7.

In [ ]:
def show_retrieval(query,k):
  retriever=vector_store.as_retriever(search_type="similarity",search_kwargs={"k":k})
  docs=retriever.invoke(query)
  print(f"\n===== k={k} | query: {query}")
  for i,d in enumerate(docs):
    first_line=d.page_content.strip().split("\n")[0][:80]
    print(f"  {i+1}. page {d.metadata['page']+1} | {first_line}")

for q in ["Why is my service returning a 504 gateway timeout?",
          "A pod keeps restarting with exit code 137"]:
  for k in (3,5,7):
    show_retrieval(q,k)

**Observation (fill in after running):** k=3 normally already contains the section that answers a single-error-code question; k=5/7 add loosely related chunks (escalation, appendix) that cost tokens and can distract the LLM. Questions that span sections (e.g. *"Which team owns the service that has DB-POOL problems and how do I escalate?"*) benefit from k=5. We use **k=3** as default and **k=5** for multi-section questions.

In [ ]:
retriever=vector_store.as_retriever(
    search_type="similarity", #similarity search between query embedding and chunk embeddings
    search_kwargs={"k":3} #top 3 chunks
)

## Part 4 — Answer generation (grounded RAG)
Prompt design:
1. **Role** – on-call Senior Site Reliability Engineer.
2. **Grounding rule** – answer *only* from the retrieved context.
3. **Refusal rule** – if the answer is not in the context, say exactly *"I couldn't find this information"*.
4. **Output format** – `1. Answer` and `2. Supporting Evidence` (verbatim quotes + page numbers), so every claim can be checked against the source.
5. `temperature=0` on the LLM to minimise hallucination.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
rag_prompt=ChatPromptTemplate.from_template(
    """
    You are a Senior Site Reliability Engineer and your job is to answer
    the user questions based ONLY on the retrieved context from the troubleshooting runbook.

    Do not use any outside knowledge.
    If the answer is not available or cannot be found in the context then say - "I couldn't find this information"

    Retrieved Context:{context}
    Question:{question}

    Provide:
    1. Answer (clear steps / bullet points; keep commands exactly as written in the context)
    2. Supporting Evidence (short verbatim quotes from the context with the page number)
    """
)

In [ ]:
from langchain_core.output_parsers import StrOutputParser

In [ ]:
rag_chain=rag_prompt|llm|StrOutputParser()

In [ ]:
def format_context(docs):
  return "\n\n".join([f"[Page {d.metadata['page']+1}]\n{d.page_content}" for d in docs])

def ask(question,k=3):
  retrieved_docs=vector_store.as_retriever(search_type="similarity",search_kwargs={"k":k}).invoke(question)
  context=format_context(retrieved_docs)
  response=rag_chain.invoke({"context":context,"question":question})
  return response,retrieved_docs

### Sample queries and generated answers

In [ ]:
question="What causes ERR-401 and how do I fix it?"
response,docs=ask(question)
print(response)

In [ ]:
question="How do I find and clear the blocking session for a DB-1205 lock wait timeout?"
response,docs=ask(question)
print(response)

In [ ]:
question="What is the acknowledgement target and update cadence for a Sev1 incident, and who is it escalated to?"
response,docs=ask(question)
print(response)

In [ ]:
question="Which team owns order-api and what is its primary datastore and critical dependency?"
response,docs=ask(question,k=5)
print(response)

### Evidence of grounding
1. Every answer cites runbook text and page numbers (Supporting Evidence section).
2. Below we print the retrieved chunks used as context for one answer so the quotes can be matched.
3. An out-of-scope question must be refused, proving the model does not hallucinate from its own knowledge.

In [ ]:
question="What is the MTTR target for Sev2 incidents?"
response,docs=ask(question)
print(response)
print("\n---- CONTEXT THE ANSWER WAS BASED ON ----")
for d in docs:
  print(f"[Page {d.metadata['page']+1}]",d.page_content[:300].replace("\n"," "),"...\n")

In [ ]:
# Out-of-scope questions -> expected: "I couldn't find this information"
for q in ["What is the capital of France?",
          "How do I configure Kubernetes autoscaling for the billing-service?"]:
  response,_=ask(q)
  print("Q:",q,"\nA:",response,"\n"+"-"*60)

## Part 5 — End-to-end RAG application
Integrates document processing (Part 1), retrieval (Parts 2–3) and LLM generation (Part 4) behind a single function and an interactive loop for real-time questions.

In [ ]:
def rag_app(question,k=3,show_sources=True):
  response,docs=ask(question,k)
  print("QUESTION:",question,"\n")
  print(response)
  if show_sources:
    print("\nSOURCES:",sorted({d.metadata['page']+1 for d in docs}))
  print("="*80)

demo_questions=[
  "What should I check first when an alert fires?",
  "How do I fix DB-POOL connection pool exhaustion?",
  "How do I renew an expired TLS certificate?",
  "Why do pods get OOMKilled and how is it fixed?",
  "What should be in the post-incident checklist?",
]
for q in demo_questions:
  rag_app(q)

In [ ]:
# Interactive real-time Q&A (type 'exit' to stop)
while True:
  q=input("Ask the runbook assistant> ").strip()
  if q.lower() in ("exit","quit",""):
    break
  rag_app(q)

## Summary
| Part | What was done |
|---|---|
| 1 | `PyPDFLoader` → 7 pages; removed repeating footers; `RecursiveCharacterTextSplitter` (1000 / 200) |
| 2 | `gemini-embedding-2` embeddings stored in FAISS with page metadata |
| 3 | Similarity retrieval, compared k = 3 / 5 / 7 → default k = 3 |
| 4 | Grounded prompt (context-only, refusal rule, Answer + Supporting Evidence), temperature 0 |
| 5 | `rag_app()` + interactive loop = working Q&A system |

**Limitations:** tables (severity matrix, service map) flatten when extracted from PDF; very short corpus means k>5 mostly adds noise.